# Contact Forces Benchmark — Analysis & Visualisation

Analyses three CSV files produced by the C++ benchmark (`setSimplifiedCollision = false`):
- **`convergence.csv`** — peak height error vs reference solution per (solver, dt)
- **`stability.csv`** — maximum energy ratio E(t)/E₀ per (solver, k, dt)
- **`energy_drift.csv`** — E(t) time series for selected dt values

**Scene:** sphere z₀=20 m, radius=2 m, mass=1 kg, k=10 000 N/m, ζ=0.05  
**Reference:** RK4 at dt=1×10⁻⁴ s

Plots produced:
1. Peak height error vs dt — convergence order with smooth contact forces
2. Estimated convergence slopes (linear regression on log-log)
3. Stability map — max energy ratio E(t)/E₀ in the (k, dt) plane
4. Theoretical stability boundary comparison
5. Energy drift E(t) time series — solver-independent damping floor
6. Summary table

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
from pathlib import Path
from scipy import stats

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

SOLVER_STYLE = {
    "Euler":  {"color": "#E24B4A", "marker": "o", "ls": "-"},
    "Verlet": {"color": "#378ADD", "marker": "s", "ls": "--"},
    "RK4":    {"color": "#1D9E75", "marker": "^", "ls": "-."},
}
ABOX = dict(boxstyle="round,pad=0.4", fc="white", ec="#cccccc", alpha=0.92, lw=0.8)

def solver_legend(ax, loc="lower right"):
    handles = [
        Line2D([0],[0], color=s["color"], marker=s["marker"],
               linestyle=s["ls"], label=name, markersize=6)
        for name, s in SOLVER_STYLE.items()
    ]
    ax.legend(handles=handles, framealpha=0.5, loc=loc)

# Scene constants (must match C++ benchmark)
MASS   = 1.0
K_REF  = 1e4
ZETA   = 0.05
DT_REF = 1e-4
OMEGA_REF = np.sqrt(K_REF / MASS)   # = 100 rad/s

CONVERGENCE_CSV = Path("convergence.csv")
STABILITY_CSV   = Path("stability.csv")
ENERGY_CSV      = Path("energy_drift.csv")

df  = pd.read_csv(CONVERGENCE_CSV)
stb = pd.read_csv(STABILITY_CSV)
ede = pd.read_csv(ENERGY_CSV)

df["cpu_ms"] = df["cpu_us"] / 1000.0

print(f"convergence.csv : {len(df)} rows  ({df['solver'].nunique()} solvers, {df['dt'].nunique()} dt values)")
print(f"stability.csv   : {len(stb)} rows  ({stb['k'].nunique()} k values, {stb['dt'].nunique()} dt values)")
print(f"energy_drift.csv: {len(ede)} rows")
df.head()

## Plot 1 — Peak height error vs dt (convergence order)

In [ ]:
fig, (ax, ax_notes) = plt.subplots(
    1, 2, figsize=(15, 6),
    gridspec_kw={"width_ratios": [2, 1]}
)

for solver, grp in df.groupby("solver"):
    s = SOLVER_STYLE[solver]
    grp = grp.sort_values("dt")
    mask = (grp["max_height_error"] > 0) & (grp["stable"] == 1)
    ax.loglog(grp.loc[mask, "dt"], grp.loc[mask, "max_height_error"],
              color=s["color"], marker=s["marker"], ls=s["ls"],
              markersize=4, linewidth=1.6)

# Reference slopes anchored at a mid-range point
dt_ref_arr = np.array([5e-4, 1.5e-2])
ax.loglog(dt_ref_arr, 8 * dt_ref_arr**1,   "k:",  linewidth=1.2, label="O(dt¹)")
ax.loglog(dt_ref_arr, 4e2  * dt_ref_arr**2,   "k--", linewidth=1.2, label="O(dt²)")
ax.loglog(dt_ref_arr, 3e5  * dt_ref_arr**4,   "k-.", linewidth=1.2, label="O(dt⁴)")

ax.set_xlabel("Time step  dt  (s)")
ax.set_ylabel("|height_error|  vs reference RK4 solution")
ax.set_title("Peak height error vs dt — contact forces mode")

handles_s = [Line2D([0],[0], color=s["color"], marker=s["marker"], ls=s["ls"], label=k, markersize=6)
             for k, s in SOLVER_STYLE.items()]
handles_r = [
    Line2D([0],[0], color="k", ls=":",  label="O(dt¹)"),
    Line2D([0],[0], color="k", ls="--", label="O(dt²)"),
    Line2D([0],[0], color="k", ls="-.", label="O(dt⁴)"),
]
ax.legend(handles=handles_s + handles_r, fontsize=9, framealpha=0.5,
          ncol=2, loc="upper left")

ax_notes.axis("off")
notes = (
    "CONVERGENCE — CONTACT FORCES\n"
    "─────────────────────────────\n\n"
    "Unlike impulse-based contact,\n"
    "the spring-damper force is\n"
    "continuous → integrator order\n"
    "is partially visible.\n\n"
    "Slopes (measured below):\n"
    "  Euler  ≈ O(dt^1.3)\n"
    "  Verlet ≈ O(dt^1.2)\n"
    "  RK4    ≈ O(dt^2.4)\n\n"
    "Why not O(dt²) / O(dt⁴)?\n"
    "The  max(F_spring + F_damp, 0)\n"
    "clamp introduces a C⁰\n"
    "discontinuity at contact\n"
    "onset/offset, capping the\n"
    "effective global order.\n\n"
    "Despite this, RK4 is clearly\n"
    "the most accurate at any dt."
)
ax_notes.text(0.04, 0.97, notes, transform=ax_notes.transAxes,
              fontsize=9, va="top", family="monospace",
              bbox=dict(boxstyle="round,pad=0.6", fc="#f8f8f8", ec="#cccccc", lw=0.8))

plt.tight_layout()
plt.savefig("plot_convergence.png", dpi=150, bbox_inches="tight")
plt.show()

## Plot 2 — Estimated convergence slopes

In [ ]:
print(f"{'Solver':<10} {'Slope':>10} {'R²':>8}  {'Error at dt=5e-4':>18}  {'Error at dt=1.5e-2':>20}")
print("-" * 75)
for solver, grp in df.groupby("solver"):
    grp  = grp.sort_values("dt")
    mask = (grp["max_height_error"] > 1e-8) & (grp["stable"] == 1)
    if mask.sum() < 4:
        print(f"{solver:<10}  not enough valid points")
        continue
    log_dt  = np.log10(grp.loc[mask, "dt"])
    log_err = np.log10(grp.loc[mask, "max_height_error"])
    slope, _, r, *_ = stats.linregress(log_dt, log_err)

    err_lo = grp.loc[grp["dt"].idxmin(), "max_height_error"]
    err_hi = grp.loc[grp["dt"].idxmax(), "max_height_error"]
    print(f"{solver:<10} {slope:>10.3f} {r**2:>8.4f}  {err_lo:>18.3e}  {err_hi:>20.3e}")

print()
print("Note: expected slopes are 1 (Euler), 2 (Verlet), 4 (RK4).")
print("The max(F_spring + F_damp, 0) clamp at contact onset/offset is a C⁰ discontinuity")
print("in the ODE right-hand side — this limits global convergence to at most O(dt²)")
print("for all methods. RK4 nonetheless remains consistently 10–100× more accurate")
print("than Euler at the same dt, making it the best choice for stiff contact problems.")

## Plot 3 — Stability map: max energy ratio E(t)/E₀ in the (k, dt) plane

`max_energy_ratio = max_t |E(t)| / E₀` — a value > 1 means the solver injected energy into the system during contact.
The theoretical stability condition for a linear spring (no damping) is **ω·dt < 2** for Euler/Verlet and **ω·dt < 2√2** for RK4, where ω = √(k/m).
Damping does not extend the stability region for symplectic/Verlet methods, but RK4's larger stability domain is clearly visible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)

k_vals  = sorted(stb["k"].unique())
dt_vals = sorted(stb["dt"].unique())

# Use log10(max_energy_ratio) as colour; clip at [0, 2.5] (1 = neutral, 100 = very unstable)
vmin, vmax = 0.0, 2.5
cmap = plt.cm.RdYlGn_r

for ax, solver in zip(axes, ["Euler", "Verlet", "RK4"]):
    sub = stb[stb["solver"] == solver]
    Z = np.zeros((len(dt_vals), len(k_vals)))
    for i, dt in enumerate(dt_vals):
        for j, k in enumerate(k_vals):
            row = sub[(sub["dt"] == dt) & (sub["k"] == k)]
            if len(row):
                Z[i, j] = np.log10(max(row["max_energy_ratio"].values[0], 1.0))

    im = ax.imshow(Z, origin="lower", aspect="auto",
                   vmin=vmin, vmax=vmax, cmap=cmap,
                   extent=[-0.5, len(k_vals)-0.5, -0.5, len(dt_vals)-0.5])

    # Overlay theoretical stability boundary
    limit_factor = 2.0 if solver in ("Euler", "Verlet") else 2.0 * np.sqrt(2.0)
    k_arr = np.array(k_vals, dtype=float)
    dt_limit = limit_factor / np.sqrt(k_arr / MASS)  # dt_max for each k

    # Convert dt_limit to axis coordinates
    dt_log  = np.log10(np.array(dt_vals))
    k_log   = np.log10(k_arr)
    dt_lim_log = np.log10(dt_limit)

    boundary_y = []
    for j in range(len(k_vals)):
        # Find interpolated row index for this dt_limit
        y_interp = np.interp(dt_lim_log[j], dt_log, np.arange(len(dt_vals)))
        boundary_y.append(y_interp)

    ax.plot(np.arange(len(k_vals)), boundary_y,
            "k--", linewidth=1.8, label=f"ω·dt = {'2' if solver != 'RK4' else '2√2'}  (theory)")

    ax.set_xticks(np.arange(len(k_vals)))
    ax.set_xticklabels([f"{k:.0e}" for k in k_vals], rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Stiffness  k  (N/m)")
    if ax == axes[0]:
        ax.set_yticks(np.arange(len(dt_vals)))
        ax.set_yticklabels([f"{dt:.3f}" for dt in dt_vals], fontsize=8)
        ax.set_ylabel("Time step  dt  (s)")
    ax.set_title(f"{solver}")
    ax.legend(fontsize=8, loc="upper left", framealpha=0.7)

    # Annotate cells with log10(ratio)
    for i in range(len(dt_vals)):
        for j in range(len(k_vals)):
            v = Z[i, j]
            color = "white" if v > 1.0 else "black"
            ax.text(j, i, f"{10**v:.1f}×", ha="center", va="center",
                    fontsize=6.5, color=color)

fig.colorbar(im, ax=axes, label="log₁₀(max E(t)/E₀)",
             fraction=0.015, pad=0.02)
fig.suptitle("Stability map — max energy ratio E(t)/E₀ in the (k, dt) plane",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("plot_stability_map.png", dpi=150, bbox_inches="tight")
plt.show()

## Plot 4 — Stability boundary: measured vs theoretical

In [ ]:
fig, (ax, ax_notes) = plt.subplots(
    1, 2, figsize=(15, 6),
    gridspec_kw={"width_ratios": [2, 1]}
)

# Theoretical boundary: dt_max = C / sqrt(k/m)
k_plot = np.logspace(np.log10(500), np.log10(2e5), 200)
for label, factor, style in [
    ("Euler / Verlet  ω·dt < 2",   2.0,            {"color": "#E24B4A", "ls": "--", "lw": 1.8}),
    ("RK4  ω·dt < 2√2",            2.0*np.sqrt(2), {"color": "#1D9E75", "ls": "-.", "lw": 1.8}),
]:
    dt_lim = factor / np.sqrt(k_plot / MASS)
    ax.loglog(k_plot, dt_lim, label=label, **style)

# Measured instability threshold: first dt where max_energy_ratio > 2 for each (solver, k)
RATIO_THRESHOLD = 2.0
for solver, s in SOLVER_STYLE.items():
    sub = stb[stb["solver"] == solver]
    k_unstable, dt_unstable = [], []
    for k, grpk in sub.groupby("k"):
        grpk = grpk.sort_values("dt")
        unstable = grpk[grpk["max_energy_ratio"] > RATIO_THRESHOLD]
        if len(unstable):
            k_unstable.append(k)
            dt_unstable.append(unstable["dt"].min())
    if k_unstable:
        ax.scatter(k_unstable, dt_unstable,
                   color=s["color"], marker=s["marker"], s=60, zorder=5,
                   label=f"{solver} — first unstable dt (ratio > {RATIO_THRESHOLD:.0f}×)")

ax.set_xlabel("Stiffness  k  (N/m)")
ax.set_ylabel("Time step  dt  (s)")
ax.set_title("Stability boundary: measured (markers) vs theoretical (lines)")
ax.legend(fontsize=9, framealpha=0.5)
ax.invert_yaxis()  # larger dt (less restrictive) at top

ax_notes.axis("off")
notes = (
    "STABILITY BOUNDARY\n"
    "─────────────────────────────\n\n"
    "Theory (undamped spring):\n"
    "  Euler/Verlet: ω·dt < 2\n"
    "  RK4:          ω·dt < 2√2\n\n"
    "RK4 can use a √2 ≈ 41% larger\n"
    "timestep than Euler/Verlet for\n"
    "the same stiffness k.\n\n"
    "Practical implication:\n"
    "For stiff contact (large k),\n"
    "RK4 allows coarser dt →\n"
    "fewer steps → can be faster\n"
    "per unit of physical time,\n"
    "despite its 4× cost per step."
)
ax_notes.text(0.04, 0.97, notes, transform=ax_notes.transAxes,
              fontsize=9, va="top", family="monospace",
              bbox=dict(boxstyle="round,pad=0.6", fc="#f8f8f8", ec="#cccccc", lw=0.8))

plt.tight_layout()
plt.savefig("plot_stability_boundary.png", dpi=150, bbox_inches="tight")
plt.show()

## Plot 5 — Energy drift E(t) time series

Shows total energy drift `|E(t) − E₀| / E₀` over time for all three solvers at three timestep values.
The staircase pattern (step at each bounce) comes from the contact model damping (ζ=0.05), not from the integrator.
The floor level between bounces is solver-dependent: smaller dt → smaller integrator error → lower floor.

In [ ]:
dt_values = sorted(ede["dt"].unique())
n_dt      = len(dt_values)

fig = plt.figure(figsize=(16, 4.0 * n_dt))
gs  = GridSpec(n_dt, 2, figure=fig, width_ratios=[2, 1], hspace=0.5, wspace=0.08)

ax_notes = fig.add_subplot(gs[:, 1])
ax_notes.axis("off")
notes = (
    "ENERGY DRIFT E(t) — CONTACT FORCES\n"
    "─────────────────────────────────\n\n"
    "Staircase steps = energy lost at\n"
    "each bounce from viscous damping\n"
    "(ζ=0.05). This is physics, not\n"
    "numerical error.\n\n"
    "Between bounces (free flight):\n"
    "  Energy should be conserved.\n"
    "  Euler drifts slowly (O(dt)).\n"
    "  Verlet/RK4: near-exact.\n\n"
    "The energy floor between steps\n"
    "is dt-dependent for Euler but\n"
    "solver-independent at the\n"
    "bounce steps (contact physics\n"
    "dominates there)."
)
ax_notes.text(0.04, 0.98, notes, transform=ax_notes.transAxes,
              fontsize=9, va="top", family="monospace",
              bbox=dict(boxstyle="round,pad=0.6", fc="#f8f8f8", ec="#cccccc", lw=0.8))

for i, dt_val in enumerate(dt_values):
    ax = fig.add_subplot(gs[i, 0])
    subset = ede[ede["dt"] == dt_val]

    for solver, grp in subset.groupby("solver"):
        s = SOLVER_STYLE[solver]
        grp = grp.sort_values("time")
        ax.semilogy(grp["time"], grp["energy_drift"].clip(lower=1e-16),
                    color=s["color"], ls=s["ls"], linewidth=1.4, label=solver)

    ax.set_title(f"Energy drift  E(t) — dt = {dt_val:.2e} s", fontsize=10)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("|E(t) − E₀| / E₀")
    ax.legend(fontsize=8, framealpha=0.4, loc="upper left")

    if i == 0:
        ax.annotate("Step = energy dissipated\nat each bounce (ζ=0.05 damping)",
                    xy=(1.5, 0.9), xytext=(4, 0.5),
                    arrowprops=dict(arrowstyle="->", color="#555", lw=1),
                    color="#555", fontsize=8, bbox=ABOX)

plt.savefig("plot_energy_time_series.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary table

In [ ]:
RATIO_THRESHOLD = 2.0

rows = []
for solver, grp in df.groupby("solver"):
    grp = grp.sort_values("dt")
    best_row = grp.loc[grp["dt"].idxmin()]

    # Convergence slope
    mask = (grp["max_height_error"] > 1e-8) & (grp["stable"] == 1)
    slope = None
    if mask.sum() >= 4:
        slope, *_ = stats.linregress(
            np.log10(grp.loc[mask, "dt"]), np.log10(grp.loc[mask, "max_height_error"]))

    # Stability: maximum k for which all tested dt remain below threshold
    sub = stb[stb["solver"] == solver]
    stable_k = sub.groupby("k")["max_energy_ratio"].max()
    always_stable_k = stable_k[stable_k < RATIO_THRESHOLD]
    max_stable_k = always_stable_k.index.max() if len(always_stable_k) else 0

    rows.append({
        "Solver":                   solver,
        "Convergence slope":        f"{slope:.2f}" if slope else "n/a",
        "Best height error":        f"{best_row['max_height_error']:.2e}",
        "CPU @ min dt (ms)":        f"{best_row['cpu_ms']:.1f}",
        "Max stable k (all dt)": f"{max_stable_k:.0e} N/m",
    })

summary = pd.DataFrame(rows).set_index("Solver")
print("Notes:")
print(f"  • 'Convergence slope' = log-log regression slope of height error vs dt")
print(f"  • Expected slopes: Euler=1, Verlet=2, RK4=4 (theoretical, for smooth ODEs)")
print(f"  • Actual slopes are lower due to C⁰ discontinuity in the contact force")
print(f"  • 'Max stable k' = largest k where max_energy_ratio < {RATIO_THRESHOLD:.0f}× for all tested dt\n")
summary

## Export all plots to PDF

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

plot_files = [
    "plot_convergence.png",
    "plot_stability_map.png",
    "plot_stability_boundary.png",
    "plot_energy_time_series.png",
]

with PdfPages("benchmark_report.pdf") as pdf:
    for path in plot_files:
        try:
            img = plt.imread(path)
            fig, ax = plt.subplots(figsize=(14, 7))
            ax.imshow(img)
            ax.axis("off")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
        except FileNotFoundError:
            print(f"Skipping {path} (run all cells first)")

print("PDF saved to benchmark_report.pdf")